# 01 USAspending pilot: PIDP, BRIC and EDA in coastal CA/OR/WA

**Task:** T1.1 (Coastal Harbor Funding Equity)

**Reads:** `data/processed/usaspending_pilot/{prime_awards,prime_transactions,subawards}.csv` and `process_log.json`, written by `uv run python -m harborfund.ingest.usaspending all`. Also reads `data/reference/coastal_counties.csv`.

**Writes:** nothing. The outputs are the pilot report, so they are kept when committed. They hold aggregates only, no raw records.

**Scope used here:**
- PIDP and EDA: an award counts if its recipient county or place-of-performance county is one of the 42 NOAA coastal shoreline counties.
- BRIC (97.047) prime awards: every prime in CA/OR/WA counts, since the recipient is usually the state. BRIC subawards are county-flagged from ZIP codes.
- Window FY2010–FY2026 (BRIC FY2020+). Award-level tables use the base action FY. Dollar tables by FY use transaction obligations by action FY.
- Dollars are nominal.

In [ ]:
import json
import numpy as np
import pandas as pd
from harborfund import config

OUT = config.PROCESSED_DIR / "usaspending_pilot"
log = json.loads((OUT / "process_log.json").read_text())
BOOL = ["in_fy_window", "has_uei", "has_duns", "recipient_is_state_gov", "prime_recipient_is_state_gov",
        "state_level_prime", "recipient_in_coastal_county", "pop_in_coastal_county", "in_coastal_county",
        "in_states", "earmark_text_flag", "earmark_candidate_year", "earmark_flag", "eda_rlf_flag",
        "harbor_kw_flag", "in_scope", "zip_multi_county", "zip_unmatched"]

def load(name):
    df = pd.read_csv(OUT / f"{name}.csv", dtype=str, low_memory=False)
    for c in BOOL:
        if c in df:
            df[c] = df[c].map({"True": True, "False": False}).fillna(False).astype(bool)
    df["fy"] = pd.to_numeric(df["fy"], errors="coerce").astype("Int64")
    if "fema_round_fy" in df:
        df["fema_round_fy"] = pd.to_numeric(df["fema_round_fy"], errors="coerce").astype("Int64")
    for c in ("fema_program_code", "fon_family", "program_basis"):
        if c in df:
            df[c] = df[c].fillna("")
    for c in [c for c in df if c.endswith("_nominal")]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ("recipient_state", "pop_state", "geo_match_basis"):
        df[c] = df[c].fillna("")
    # State the row is counted in: the matched coastal county's state; otherwise
    # (state-level BRIC primes) the recipient state if in CA/OR/WA, else the POP state.
    rec_ok = df["recipient_state"].isin(["CA", "OR", "WA"])
    df["geo_state"] = np.select(
        [df["recipient_in_coastal_county"], df["pop_in_coastal_county"], rec_ok],
        [df["recipient_state"], df["pop_state"], df["recipient_state"]], default=df["pop_state"])
    return df

awards, txns, subs = (load(n) for n in ("prime_awards", "prime_transactions", "subawards"))
print("raw run:", log["raw_run"], "| processed:", log["processed_utc"])
pd.DataFrame({k: v["raw_rows"] | {"kept_" + kk: vv for kk, vv in v["in_scope_rows"].items()}
              for k, v in log["downloads"].items()}).T

## 1. Scope and award types

In [ ]:
A = awards[awards.in_scope & awards.in_fy_window].copy()
T = txns[txns.in_scope & txns.in_fy_window].copy()
S = subs[subs.in_fy_window].copy()
print(f"prime awards in scope: {len(A):,} | transactions: {len(T):,} | subawards in CA/OR/WA: {len(S):,} "
      f"(in coastal counties: {S.in_coastal_county.sum():,})")
pd.crosstab(A["program_id"], A["assistance_type_description"].fillna("(blank)"), margins=True)

## 2. Row counts and nominal totals by program × state × FY
Awards are counted by base action FY, and the dollars are each award's total obligation to date. The second table sums transaction obligations by action FY.

In [ ]:
t_awards = (A.groupby(["program_id", "geo_state", "fy"])
              .agg(awards=("assistance_award_unique_key", "nunique"), total_obligated_nominal=("total_obligated_nominal", "sum"))
              .reset_index())
pd.set_option("display.max_rows", 400)
t_awards.pivot_table(index=["program_id", "geo_state"], columns="fy", values="awards", fill_value=0, aggfunc="sum")

In [ ]:
(T.groupby(["program_id", "geo_state", "fy"])["federal_action_obligation_nominal"].sum()
   .unstack("fy").fillna(0).div(1e6).round(2))  # $M nominal, by action FY

In [ ]:
summary = (A.groupby(["program_id", "geo_state"])
             .agg(awards=("assistance_award_unique_key", "nunique"),
                  obligated_m=("total_obligated_nominal", lambda s: s.sum() / 1e6))
             .round(2))
summary

## 3. Recipient identifiers (UEI / legacy DUNS)
SAM.gov switched from DUNS to UEI in April 2022, so older awards may have only a DUNS.

In [ ]:
ids = A.assign(uei_or_duns=A.has_uei | A.has_duns)
pd.concat([
    ids.groupby("program_id")[["has_uei", "has_duns", "uei_or_duns"]].mean().round(3),
    ids.groupby("program_id").size().rename("awards"),
], axis=1)

In [ ]:
ids.assign(era=pd.cut(ids.fy.astype(float), [2009, 2021, 2026], labels=["FY2010-21", "FY2022-26"])) \
   .groupby(["program_id", "era"], observed=True)[["has_uei", "has_duns", "uei_or_duns"]].mean().round(3)

In [ ]:
sub_ids = S[S.in_coastal_county]
sub_ids.groupby("program_id")[["has_uei", "has_duns"]].mean().round(3).assign(subawards=sub_ids.groupby("program_id").size())

## 4. BRIC: state-held prime dollars and what subawards identify

In [ ]:
B = A[A.aln == "97.047"]
bric_state = B.groupby("recipient_is_state_gov")["total_obligated_nominal"].agg(["count", "sum"])
bric_state["share_of_dollars"] = (bric_state["sum"] / bric_state["sum"].sum()).round(3)
bric_state

In [ ]:
state_primes = B[B.recipient_is_state_gov]
sb = subs[(subs.aln == "97.047") & subs.prime_award_unique_key.isin(state_primes.assistance_award_unique_key)]
rows = []
for st, g in state_primes.groupby("recipient_state"):
    s_all = sb[sb.prime_award_unique_key.isin(g.assistance_award_unique_key)]
    s_coast = s_all[s_all.in_coastal_county]
    prime_d = g.total_obligated_nominal.sum()
    rows.append({"state": st, "state_primes": len(g), "prime_obligated_m": prime_d / 1e6,
                 "subawards": len(s_all), "subaward_m": s_all.subaward_amount_nominal.sum() / 1e6,
                 "share_identified_by_subawards": s_all.subaward_amount_nominal.sum() / prime_d if prime_d else None,
                 "coastal_subawards": len(s_coast), "coastal_subaward_m": s_coast.subaward_amount_nominal.sum() / 1e6})
pd.DataFrame(rows).round(3)

Caveat: subaward amounts are FFATA-reported and can be double-reported or amended, so the identified share may go above 1. The prime total is the obligation to date.

In [ ]:
sub_geo = S[S.aln == "97.047"]
pd.Series({"subawards_in_states": len(sub_geo), "zip_unmatched": sub_geo.zip_unmatched.sum(),
           "zip_multi_county": sub_geo.zip_multi_county.sum(), "in_coastal_county": sub_geo.in_coastal_county.sum()})

## 5. Program split and earmark flags
- **97.047** is classified from the FEMA award ID, e.g. `EMS-2022-PD-0002` → round FY2022, code `PD`. `BR` is BRIC, `PC` is legacy PDM competitive, and `PD` from FY2022 is the PDM congressionally directed (earmark) award. Descriptions aren't used because they're unreliable: PDM earmark awards are often described as "BRIC", and one is labelled "Emergency Management Performance Grants".
- **20.823** is classified from the funding-opportunity family. `MA-PID-yy` is a PIDP round. `MARAD BAA 2019/2020` covers awards from 2019/2020 broad agency announcements, probably the early PIDP rounds (that mapping is unverified). Marine Highway (AMHP) and DOT BUILD/INFRA records are split off as `other_20823`.
- No 20.823 record says Community Project Funding, so PIDP earmarks (FY2024, FY2026) **can't** be separated with USAspending alone. T2.4's THUD tables are needed for that.

In [ ]:
E = A[A.aln.isin(["20.823", "97.047"])]
(E.groupby(["aln", "program_id", "program_basis"])
   .agg(awards=("assistance_award_unique_key", "nunique"), obligated_m=("total_obligated_nominal", lambda s: s.sum() / 1e6),
        earmark_flagged=("earmark_flag", "sum"))
   .round(2))

In [ ]:
F = awards[awards.aln == "97.047"]
pd.crosstab([F.fema_program_code.replace("", "?"), F.in_scope], F.fema_round_fy.astype("string").fillna("?"))

In [ ]:
P = awards[awards.aln == "20.823"]
pd.crosstab(P.fon_family, P.fy)

In [ ]:
E[E.earmark_flag][["aln", "program_id", "award_id_fain", "fy", "recipient_name", "geo_state", "in_coastal_county",
                   "total_obligated_nominal", "prime_award_base_transaction_description"]].sort_values(["aln", "fy"])

In [ ]:
# Subawards under PDM earmark primes carry LPDM project IDs, which corroborates the FAIN rule
sub_cpf = subs[subs.program_id == "pdm_cpf"]
pd.Series({"subawards": len(sub_cpf),
           "with LPDM project id": sub_cpf.subaward_description.fillna("").str.contains(r"^LPDM\d{2}", regex=True).sum(),
           "in coastal county": sub_cpf.in_coastal_county.sum(),
           "coastal $M": round(sub_cpf.loc[sub_cpf.in_coastal_county, "subaward_amount_nominal"].sum() / 1e6, 2)})

## 6. Harbor keyword flag and geography match basis
`harbor_kw_flag` applies only the inventory's keyword rule, matched at word starts (so 'port' doesn't match 'support'). The crosswalk part of the rule comes later (T1.2/T1.3). PIDP is `all` (a port program). BRIC/PDM prime descriptions are generic program text, so the keyword rule can only work on subaward descriptions, and OpenFEMA (T2.5a) has the project titles.

In [ ]:
pd.concat([A.groupby("program_id")["harbor_kw_flag"].agg(["sum", "mean"]).rename(columns={"sum": "flagged", "mean": "share"}),
           A[A.harbor_kw_flag].groupby("program_id")["total_obligated_nominal"].sum().div(1e6).rename("flagged_m")], axis=1).round(3)

In [ ]:
pd.crosstab(A.program_id, A.geo_match_basis.replace("", "(state-level BRIC prime)"), margins=True)

In [ ]:
if "eda_rlf_flag" in A:
    print(A[A.aln.isin(["11.300", "11.307"])].groupby("program_id")["eda_rlf_flag"].sum())

## 7. Independent count checks
One program-year per program, compared with a published figure. All checks use every award in the state, not just coastal counties, because the published figures are statewide.

| Program | External figure | Source (read 2026-11-18) |
| --- | --- | --- |
| PIDP | MARAD FY2022 PIDP award list: 12 awards in CA/OR/WA | https://www.maritime.dot.gov/sites/marad.dot.gov/files/2022-10/FY%202022%20Port%20Infrastructure%20Development%20Grant%20Awards.pdf |
| EDA Public Works | EDA FY2014 annual report, Appendix 2: CA 2 / $5,620,500; OR 1 / $1,580,800; WA 4 / $5,400,000 | https://eda.gov/sites/default/files/2024-11/Appendix-2-fy2014-Summary-of-EDA-Fiscal-Investments-by-State.pdf |
| EDA EAA (reference only) | Same appendix: CA 5 / $1,985,432; OR 3 / $901,036; WA 3 / $1,721,415 | same |
| BRIC | Oregon OEM, March 2026: "Oregon's single awarded FY2020 project" | https://apps.oregon.gov/oregon-newsroom/OR/OEM/Posts/Post/BRIC-update-March-2026 |

The MARAD figures were read through a text-extraction tool, so the project names and amounts below are as extracted.

In [ ]:
marad_fy22 = pd.DataFrame([
    ("CA", "Fisherman's Terminal Piling Replacement (Eureka)", 650000),
    ("CA", "Middle Harbor Terminal Zero Emission Conversion (Long Beach)", 30141080),
    ("CA", "Outer Harbor Terminal Redevelopment (Oakland)", 36592875),
    ("CA", "Amador Street Infrastructure Improvement (San Francisco)", 9607500),
    ("CA", "Port of Stockton Rail Rehabilitation & Upgrade", 9628477),
    ("CA", "Seawall Replacement (Crescent City)", 7366566),
    ("OR", "Terminal 6 Infrastructure Improvements (Portland)", 24360000),
    ("WA", "Terminal 5 Export, Expansion, and Emissions Reduction (Seattle)", 17035900),
    ("WA", "Grays Harbor Terminal 4 Expansion & Redevelopment (Aberdeen)", 25500000),
    ("WA", "Intermodal Handling and Transfer Facility Improvements (Port Angeles)", 8608000),
    ("WA", "Master Plan, Pier Design, and Boat Ramp Construction (La Conner)", 11073627),
    ("WA", "Seaport Throughput Improvement (Olympia)", 9270918),
], columns=["state", "project", "marad_amount"])
us22 = awards[(awards.fon_family == "MA-PID-22")][["recipient_name", "recipient_state", "in_coastal_county", "total_obligated_nominal"]]
chk = marad_fy22.merge(us22, left_on="marad_amount", right_on="total_obligated_nominal", how="outer", indicator=True)
print("MARAD FY2022 CA/OR/WA:", len(marad_fy22), "| USAspending MA-PID-22 in CA/OR/WA:", len(us22),
      "| exact-amount matches:", (chk._merge == "both").sum())
chk.sort_values(["_merge", "state"])

Unmatched rows, as of the 2026-11-18 pull:
- **Swinomish (La Conner, $11,073,627)** is in the national file, but USAspending gives the recipient and the place of performance as Superior, WI. No address filter can find it, and a UEI crosswalk would. That's the crosswalk-first rule working as intended.
- **Seattle T5** shows $17,308,440 against MARAD's $17,035,900, and **Oakland** $35,467,875 against $36,592,875. These are the same projects with amended obligations. The difference is $272,540 for Seattle and $1,125,000 for Oakland.
- **Oakland $5.2M** (substation, FY2024) carries a `MA PID-22 001` number but isn't on MARAD's FY2022 list. The number may be a typo for another round. Not checked.

In [ ]:
eda14 = (awards[(awards.aln.isin(["11.300", "11.307"])) & (awards.fy == 2014) & awards.recipient_state.isin(["CA", "OR", "WA"])]
         .groupby(["program_id", "recipient_state"])["total_obligated_nominal"].agg(["count", "sum"]))
ext = pd.DataFrame([("eda_public_works", "CA", 2, 5620500), ("eda_public_works", "OR", 1, 1580800),
                    ("eda_public_works", "WA", 4, 5400000), ("eda_eaa", "CA", 5, 1985432),
                    ("eda_eaa", "OR", 3, 901036), ("eda_eaa", "WA", 3, 1721415)],
                   columns=["program_id", "recipient_state", "eda_count", "eda_sum"])
eda14.reset_index().merge(ext, on=["program_id", "recipient_state"], how="outer")

In [ ]:
or20 = awards[(awards.aln == "97.047") & (awards.fema_program_code == "BR") & (awards.fema_round_fy == 2020) & (awards.recipient_state == "OR")]
print("Oregon FY2020-round BRIC prime awards in USAspending:", len(or20), "| Oregon OEM: 1")
or20[["award_id_fain", "recipient_name", "total_obligated_nominal"]]